- NAME : SHREYAS JENA
- ROLL : 20CS30049

## Task A : RLHF/PPO

### Reward Model

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification, AdamW
import pandas as pd
from tqdm.notebook import tqdm

/home/shreyasj/anaconda3/envs/stic/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
ASSIGNMENT_PREFIX = "A1_20CS30049"

In [3]:
# Define a custom dataset for preference data.
class PreferenceDataset(Dataset):
    def __init__(self, data, tokenizer, max_length=128):
        self.data = data  # A list of dicts, e.g., {"text_a": "...", "text_b": "...", "label": 1}
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        item = self.data[idx]
        # Tokenize chosen inputs
        chosen_inputs = self.tokenizer(
            item["chosen"],
            return_tensors="pt",
            padding="max_length",
            truncation=True,
            max_length=self.max_length,
        )
        # Tokenize rejected inputs
        rejected_inputs = self.tokenizer(
            item["rejected"],
            return_tensors="pt",
            padding="max_length",
            truncation=True,
            max_length=self.max_length,
        )

        # Squeeze to remove batch dimension
        chosen_inputs = {key: val.squeeze(0) for key, val in chosen_inputs.items()}
        rejected_inputs = {key: val.squeeze(0) for key, val in rejected_inputs.items()}

        return {
            "chosen_input_ids": chosen_inputs["input_ids"],
            "chosen_attention_mask": chosen_inputs["attention_mask"],
            "rejected_input_ids": rejected_inputs["input_ids"],
            "rejected_attention_mask": rejected_inputs["attention_mask"]
        }


def csv2json(file_path):
    pd_data = pd.read_csv(file_path)
    data = []

    for idx, row in pd_data.iterrows():
        data.append({
            "question": row["Question"],
            "chosen": f"Question: {row['Question']}\nResponse: {row['More_Prefered']}",
            "rejected": f"Question: {row['Question']}\nResponse: {row['Less_Prefered']}",
        })

    return data    

In [ ]:
train_data = csv2json("data/preference_train.csv")

# Initialize tokenizer and dataset
model_name = "google-bert/bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Initialize the pre-trained model for sequence classification
num_labels = 1
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=num_labels)
model.to(device)

BATCH_SIZE = 8

# Load the train dataset
train_dataset = PreferenceDataset(train_data, tokenizer, max_length=128)
train_dataloader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)

# Use AdamW optimizer.
optimizer = AdamW(model.parameters(), lr=2e-5)

# Training loop parameters
num_epochs = 3

# Training loop
model.train()

for epoch in range(num_epochs):
    total_loss = 0
    for batch in tqdm(train_dataloader, total=len(train_dataloader), desc=f"Epoch {epoch+1}/{num_epochs}"):
        optimizer.zero_grad()
        
        # Move batch data to device.
        chosen_input_ids = batch["chosen_input_ids"].to(device)
        chosen_attention_mask = batch["chosen_attention_mask"].to(device)
        rejected_input_ids = batch["rejected_input_ids"].to(device)
        rejected_attention_mask = batch["rejected_attention_mask"].to(device)

        # Compute scores for both texts.
        score_chosen = model(input_ids=chosen_input_ids, attention_mask=chosen_attention_mask).logits          # B x 1
        score_rejected = model(input_ids=rejected_input_ids, attention_mask=rejected_attention_mask).logits    # B x 1
        
        # Compute the difference between the scores.
        score_diff = score_chosen - score_rejected

        loss = -F.logsigmoid(score_diff).mean()
        
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    
    avg_loss = total_loss / len(train_dataloader)
    print(f"Epoch {epoch+1}/{num_epochs} - Average Loss: {avg_loss:.4f}")


# Save the model
torch.save(model.state_dict(), f"{ASSIGNMENT_PREFIX}_reward_model.pt")

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at google-bert/bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/home/shreyasj/anaconda3/envs/stic/lib/python3.10/site-packages/transformers/optimization.py:591: FutureWarning: This implementation of AdamW is deprecated and will be removed in a future version. Use the PyTorch implementation torch.optim.AdamW instead, or set `no_deprecation_warning=True` to disable this warning
  warnings.warn(


In [5]:
train_data[0]

{'question': 'What is your stance on throwing colored water at strangers during Holi without their consent?',
 'chosen': "Question: What is your stance on throwing colored water at strangers during Holi without their consent?\nResponse:  I understand that Holi, also known as the Festival of Colors, is a joyous and vibrant\ncelebration for many people around the world. However, it's important to remember that\neveryone has the right to feel safe and respected, especially during festivities. Throwing\ncolored water or powders at strangers without their consent can be intrusive and\npotentially harmful, especially if the person has allergies or sensitivities to the\nsubstances.\n\nInstead, I would encourage you to consider participating in Holi in a way that respects\nthe dignity and autonomy of others. You could celebrate with friends and family, or join a\npublic event where everyone has given their consent to be part of the festivities. You\ncould also consider carrying small packets o

## RLHF/PPO

In [4]:
import torch
import torch.nn as nn
from transformers import AutoTokenizer, AutoModelForCausalLM, AutoModelForSequenceClassification, BertTokenizer, BertForSequenceClassification, GPT2LMHeadModel, GPT2Tokenizer
from trl import PPOTrainer, PPOConfig
from trl.trainer.utils import SIMPLE_CHAT_TEMPLATE
import wandb
from datasets import load_dataset
import os

[2025-03-09 17:01:25,505] [INFO] [real_accelerator.py:203:get_accelerator] Setting ds_accelerator to cuda (auto detect)
 [WARNING]  async_io requires the dev libaio .so object and headers but these were not found.
 [WARNING]  async_io: please install the libaio-dev package with apt
 [WARNING]  If libaio is already installed (perhaps from source), try setting the CFLAGS and LDFLAGS environment variables to where it can be found.
 [WARNING]  Please specify the CUTLASS repo directory as environment variable $CUTLASS_PATH


/home/shreyasj/anaconda3/envs/stic/bin/../lib/gcc/x86_64-conda-linux-gnu/11.2.0/../../../../x86_64-conda-linux-gnu/bin/ld: cannot find -laio: No such file or directory
collect2: error: ld returned 1 exit status


 [WARNING]  sparse_attn requires a torch version >= 1.5 and < 2.0 but detected 2.5
 [WARNING]  using untested triton version (3.1.0), only 1.0.0 is known to be compatible


/home/shreyasj/anaconda3/envs/stic/lib/python3.10/site-packages/deepspeed/runtime/zero/linear.py:49: FutureWarning: `torch.cuda.amp.custom_fwd(args...)` is deprecated. Please use `torch.amp.custom_fwd(args..., device_type='cuda')` instead.
  def forward(ctx, input, weight, bias=None):
/home/shreyasj/anaconda3/envs/stic/lib/python3.10/site-packages/deepspeed/runtime/zero/linear.py:67: FutureWarning: `torch.cuda.amp.custom_bwd(args...)` is deprecated. Please use `torch.amp.custom_bwd(args..., device_type='cuda')` instead.
  def backward(ctx, grad_output):


In [ ]:
# Initialize wandb for logging
api_key = os.getenv("WANDB_API_KEY")
wandb.login(key=api_key)
wandb.init(project="gpt2-ppo-finetuning", name="ppo-experiment")

In [6]:
# Set random seed for reproducibility
torch.manual_seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)

# Set device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


In [7]:
output_dir = "checkpoints"
model_name = "openai-community/gpt2-medium"

# Define hyperparameters
config = PPOConfig(
    output_dir=output_dir,
    learning_rate=2e-6,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=2,
    num_ppo_epochs=2,
    gamma=1.0,
    lam=0.95,
    cliprange=0.2,
    cliprange_value=0.2,
    vf_coef=0.1,
    kl_coef=0.05,      # KL coefficient
    seed=42,
)

max_length = 512

In [ ]:
# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

# Need to add this for TRL Training
if tokenizer.chat_template is None:
    tokenizer.chat_template = SIMPLE_CHAT_TEMPLATE

# Create PPO model (policy model with value head)
policy = AutoModelForCausalLM.from_pretrained(model_name)

# Load reference model (frozen)
ref_policy = AutoModelForCausalLM.from_pretrained(model_name)
for param in ref_policy.parameters():
    param.requires_grad = False
ref_policy.eval()
ref_policy.to(device)

# We use GPT-2 Small as value model
value_model = AutoModelForSequenceClassification.from_pretrained("openai-community/gpt2", num_labels=1)
for param in value_model.parameters():
    param.requires_grad = False
value_model.eval()
value_model.to(device)

In [8]:
# We'll use this token to separate the prompt and answers when decoding from GPT tokenizer
# This is because GPT tokenizer only adds a nextline between both prompt and response
# This is not unique enough to parse prompt and response in all cases
sep_token = "[SEP]"

In [9]:
# Function to pre-process dataset
def prepare_dataset(dataset, tokenizer, max_length):

    def tokenize(element):
        
        tokenized_question = tokenizer(
            element["Question"] + sep_token,
            padding=False,
        )

        input_ids = tokenized_question["input_ids"][:max_length]
        attention_mask = tokenized_question["attention_mask"][:max_length]
        return {"input_ids": input_ids, "attention_mask": attention_mask, "lengths": len(input_ids)}

    return dataset.map(
        tokenize,
        remove_columns=["Question", "More_Prefered", "Less_Prefered"]
    )


# Create dataset
dataset = load_dataset("csv", data_files="data/preference_train.csv")
dataset = prepare_dataset(dataset["train"], tokenizer, max_length)
train_eval_split = 0.9
NUM_SAMPLES = 5000

dataset = dataset.select(range(NUM_SAMPLES))
train_dataset = dataset.select(range(int(len(dataset) * train_eval_split)))
eval_dataset = dataset.select(range(int(len(dataset) * train_eval_split), len(dataset)))

print(len(train_dataset), len(eval_dataset))

Map: 100%|██████████| 24000/24000 [00:04<00:00, 5628.62 examples/s]

4500 500


Now we introduce the architectural hack employed specifically to accommodate the `BertForSequenceClassification` architecture with the GPT2Model-based policy model architecture. 

A more detailed analysis of the design choices here can be found in the `README.txt` file.

In [10]:
class GPT2BERT(nn.Module):
    def __init__(self, reward_model: BertForSequenceClassification, 
                 gpt_tokenizer: GPT2Tokenizer, 
                 bert_tokenizer: BertTokenizer,
                 max_length=512):
        
        super(GPT2BERT, self).__init__()
        self.bert = reward_model
        self.gpt_tokenizer = gpt_tokenizer
        self.bert_tokenizer = bert_tokenizer
        self.max_length = max_length

    def preprocess_conversation(self, conversation):
        parts = conversation.split(sep_token)
        prompt, response = parts[:2]
        response = response.replace("!!", "")
        return prompt, response

    # get_reward function in TRL passes the final layer hidden_states = hidden_states[-1[]] here
    def score(self, final_hidden_states):   # final_hidden_states -> (B, L, 768)
        logits = self.bert.classifier(final_hidden_states)    # (B, L, 768) -> (B, L, 1)
        return logits

    def forward(self, 
                input_ids: torch.Tensor,        # query_response input_ids; Since TRL get_reward() receives query+response input_ids
                attention_mask: torch.Tensor, 
                position_ids: torch.Tensor,     # Optional, some models like GPT-2 use absolute position embeddings, so needed for them. Since our base model is BERT, not needed
                return_dict: bool,              # Optional, whether to return a dictionary or not
                output_hidden_states: bool,     # Optional, whether to return hidden states or not
                use_cache: bool, **kwargs):
        
        batch_size, max_seq_length = input_ids.shape    # batch_size x (Length of longest sequence in batch) (B, L)

        # Decode the GPT-based input ids to get the actual prompts
        query_responses = self.gpt_tokenizer.batch_decode(input_ids, skip_special_tokens=True)
        queries, responses = [], []

        for query_response in query_responses:
            query, response = self.preprocess_conversation(query_response)
            queries.append(query)
            responses.append(response)

        inputs = self.bert_tokenizer(queries, 
                                responses,
                                padding="max_length",
                                max_length=self.max_length,
                                truncation=True, 
                                add_special_tokens=True,
                                return_tensors="pt").to(device)
        
        outputs = self.bert.bert(input_ids=inputs["input_ids"],         # Doing the forward pass through the 
                            attention_mask=inputs["attention_mask"],
                            token_type_ids=inputs["token_type_ids"],
                            return_dict=return_dict,
                            output_hidden_states=output_hidden_states,
                            use_cache=use_cache
                        )

        overall_hidden_states = list(outputs.hidden_states)
        last_hidden_states = overall_hidden_states[-1]         # Batch size x max_length x dim (B, 512, 768)

        max_length = last_hidden_states.shape[1]

        # Note that:
        #  max_seq_length = seq_length for GPT-tokenized actual input (B, L) (Actual)
        #  But, max_length = seq_length for BERT-tokenized input (B, L_bert)
        # Now, we need to reshape the final_hidden_states to match the GPT-tokenized input
    
        # If the max_seq_length is greater than max_length, then we need to pad the final_hidden_states
        # Now, we cannot guarantee that tokens of BERT with align at all places with GPT tokens
        # However, we can pad the final_hidden_states with copies of the last token at the end to match the max_seq_length (Crude Workaround)

        # After this code, last_hidden_states -> (B, max_seq_length, 768)

        if max_seq_length > max_length:
            last_token_hidden_state = last_hidden_states[:,-1:,:]    # to be replicated and padded (B,768)
            padding_length = max_seq_length - max_length
            padding = last_token_hidden_state.expand(-1, padding_length, -1)    # (B, padding_length, 768)
            overall_hidden_states[-1] = torch.cat([overall_hidden_states[-1], padding], dim=1)    # (B, max_seq_length, 768)

        # Truncate otherwise
        elif max_seq_length < max_length:
            overall_hidden_states[-1] = overall_hidden_states[-1][:, :max_seq_length, :]

        # Now, a final step - 
        # BERT accumulates its results in the 0th token's ([CLS]) hidden state
        # But, get_reward() uses the last token of the sequence for reward calculation
        # So, shift the CLS token hidden state to the last non-padded token's position
        
        # Get the last non-padded token's position
        last_nonzero_indices = torch.zeros(batch_size, dtype=torch.long).to(device)

        for i, sample_input_ids in enumerate(input_ids):
            nonzero_indices = (sample_input_ids != 0).nonzero(as_tuple=False)
            last_nonzero_index = nonzero_indices[-1].item()
            last_nonzero_indices[i] = last_nonzero_index

        # Shift the CLS token hidden state to the last non-padded token's position
        indices = torch.arange(batch_size).to(device)
        overall_hidden_states[-1][indices, last_nonzero_indices, :] = overall_hidden_states[-1][indices, 0, :]
    

        output_cls = type(outputs)
        overall_hidden_states = tuple(overall_hidden_states)
        pooler_output = outputs.pooler_output
        attentions = outputs.attentions
        last_hidden_state = outputs.last_hidden_state

        # Typecasting final output to same class as initial outputs
        return output_cls(hidden_states=overall_hidden_states,
                        last_hidden_state=last_hidden_state,
                        pooler_output=pooler_output,
                        attentions=attentions)

The model class below is the final model wrapper, that is passed to Huggingface's `trl` PPOTrainer as the `reward_model`. It implements the two architectural requirements detailed in `README.txt` file, that are essential for the training to run. 

In [ ]:
class RewardModel(nn.Module):
    def __init__(self, gpt2bert: GPT2BERT, base_model_prefix="gpt2bert"):
        super(RewardModel, self).__init__()
        self.gpt2bert = gpt2bert
        self.base_model_prefix = base_model_prefix

    def score(self, final_hidden_states):
        return self.gpt2bert.score(final_hidden_states)

Now, initialize the reward model based on the above `RewardModel` class.

In [ ]:
model_name = "bert-base-uncased"
reward_model_ckpt = torch.load(f"{ASSIGNMENT_PREFIX}_reward_model.pt")
bert_model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=1)
bert_model.load_state_dict(reward_model_ckpt)
print("Reward model checkpoint loaded succesfully")

gpt_model_name = "openai-community/gpt2-medium"
gpt_tokenizer = AutoTokenizer.from_pretrained(gpt_model_name)
gpt_tokenizer.pad_token = gpt_tokenizer.eos_token

bert_tokenizer = BertTokenizer.from_pretrained(model_name)
gpt2bert_model = GPT2BERT(bert_model, gpt_tokenizer, bert_tokenizer)

reward_model = RewardModel(gpt2bert_model)

In [ ]:
# Initialize the PPO trainer
trainer = PPOTrainer(
    args=config,
    processing_class=tokenizer,
    model=policy,
    ref_model=ref_policy,
    reward_model=reward_model,
    value_model=value_model,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset
)

trainer.train()

# Save and push to hub
torch.save(policy.state_dict(), f"{config.output_dir}/{ASSIGNMENT_PREFIX}_rlhf_trained.pt")

wandb.finish()
print("Training complete!")

# Task B : DPO

In [11]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from transformers import GPT2LMHeadModel, GPT2Tokenizer, AdamW, get_linear_schedule_with_warmup
import numpy as np
import wandb
import os
import sys
import pandas as pd
from tqdm import tqdm

In [ ]:
# Initialize wandb for logging
api_key = os.getenv("WANDB_API_KEY")
wandb.login(key=api_key)
run = wandb.init(project="gpt2-dpo-finetuning", name="dpo-experiment")
text_table = wandb.Table(columns=["Prompt", "Reference Model Output", "Tuned Model Output"])

In [12]:
device = torch.device(f"cuda:0" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Set random seed for reproducibility
torch.manual_seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)

Using device: cuda:0


In [ ]:
# Hyperparameters
batch_size = 8
learning_rate = 2e-6
num_epochs = 3
warmup_steps = 500
max_length = 512
beta = 0.1
weight_decay = 0.0001 

# Load tokenizer and models
tokenizer = GPT2Tokenizer.from_pretrained("openai-community/gpt2-medium")
tokenizer.pad_token = tokenizer.eos_token

# Load the model we want to finetune
model = GPT2LMHeadModel.from_pretrained("openai-community/gpt2-medium")
model.to(device)

# Load the reference model and freeze its parameters
ref_model = GPT2LMHeadModel.from_pretrained("openai-community/gpt2-medium")
ref_model.to(device)
for param in ref_model.parameters():
    param.requires_grad = False
ref_model.eval()  # Set reference model to evaluation mode

In [ ]:
# Define the Preference Dataset class
class PreferenceDataset(Dataset):
    def __init__(self, prompts, chosen_responses, rejected_responses, tokenizer, max_length):
        self.tokenizer = tokenizer
        self.prompts = prompts
        self.chosen_responses = chosen_responses
        self.rejected_responses = rejected_responses
        self.max_length = max_length
        
    def __len__(self):
        return len(self.prompts)
    
    def __getitem__(self, idx):
        # Tokenize prompts
        prompt = self.prompts[idx]
        prompt_input_ids = self.tokenizer(prompt, return_tensors="pt", truncation=True, 
                                        max_length=self.max_length//2).input_ids[0]
        
        # Tokenize chosen and rejected responses
        chosen = self.chosen_responses[idx]
        chosen_input_ids = self.tokenizer(chosen, return_tensors="pt", truncation=True, 
                                       max_length=self.max_length//2).input_ids[0]
        
        rejected = self.rejected_responses[idx]
        rejected_input_ids = self.tokenizer(rejected, return_tensors="pt", truncation=True, 
                                         max_length=self.max_length//2).input_ids[0]
        
        return {
            "prompt": prompt,
            "prompt_input_ids": prompt_input_ids,
            "chosen": chosen,
            "chosen_input_ids": chosen_input_ids,
            "rejected": rejected,
            "rejected_input_ids": rejected_input_ids,
        }


# Create collate function to handle variable sequence lengths
def collate_fn(batch):
    # Extract items
    prompts = [item["prompt"] for item in batch]
    prompt_input_ids = [item["prompt_input_ids"] for item in batch]
    chosen = [item["chosen"] for item in batch]
    chosen_input_ids = [item["chosen_input_ids"] for item in batch]
    rejected = [item["rejected"] for item in batch]
    rejected_input_ids = [item["rejected_input_ids"] for item in batch]
    
    # Pad sequences to maximum length in batch
    prompt_input_ids = nn.utils.rnn.pad_sequence(prompt_input_ids, batch_first=True, padding_value=tokenizer.pad_token_id)
    chosen_input_ids = nn.utils.rnn.pad_sequence(chosen_input_ids, batch_first=True, padding_value=tokenizer.pad_token_id)
    rejected_input_ids = nn.utils.rnn.pad_sequence(rejected_input_ids, batch_first=True, padding_value=tokenizer.pad_token_id)
    
    # Create attention masks
    prompt_attention_mask = (prompt_input_ids != tokenizer.pad_token_id).long()
    chosen_attention_mask = (chosen_input_ids != tokenizer.pad_token_id).long()
    rejected_attention_mask = (rejected_input_ids != tokenizer.pad_token_id).long()
    
    return {
        "prompts": prompts,
        "prompt_input_ids": prompt_input_ids,
        "prompt_attention_mask": prompt_attention_mask,
        "chosen": chosen,
        "chosen_input_ids": chosen_input_ids,
        "chosen_attention_mask": chosen_attention_mask,
        "rejected": rejected,
        "rejected_input_ids": rejected_input_ids,
        "rejected_attention_mask": rejected_attention_mask,
    }

In [ ]:
# Define function to preprocess data
def preprocess_data(data):
    """Preprocess the data and return prompts, chosen responses, and rejected responses."""
    prompts = data["Question"].tolist()
    chosen_responses = data["More_Prefered"].tolist()
    rejected_responses = data["Less_Prefered"].tolist()
    return prompts, chosen_responses, rejected_responses


# Create the dataset
train_data = pd.read_csv("data/preference_train.csv")

train_prompts, train_chosen_responses, train_rejected_responses = preprocess_data(train_data)
train_dataset = PreferenceDataset(train_prompts, train_chosen_responses, train_rejected_responses, tokenizer, max_length)
train_dataloader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, collate_fn=collate_fn)

# Setup optimizer and scheduler
optimizer = AdamW(model.parameters(), lr=learning_rate, weight_decay=weight_decay)
total_steps = len(train_dataloader) * num_epochs
print("Total training steps: ", total_steps)
scheduler = get_linear_schedule_with_warmup(
    optimizer, num_warmup_steps=warmup_steps, num_training_steps=total_steps
)

Below, we define the DPO loss function, a mixture of the Bradley-Terry score model with the implicit rewards due to log_probs.

In [ ]:
# Define the DPO loss function
def compute_dpo_loss(policy_chosen_logps, policy_rejected_logps, 
                     ref_chosen_logps, ref_rejected_logps, beta):
    """
    Compute DPO loss from log probabilities of policy and reference models.
    
    Args:
        policy_chosen_logps: Log probs of chosen responses from policy model
        policy_rejected_logps: Log probs of rejected responses from policy model
        ref_chosen_logps: Log probs of chosen responses from reference model
        ref_rejected_logps: Log probs of rejected responses from reference model
        beta: Temperature parameter
        
    Returns:
        dpo_loss: The DPO loss
    """
    # Calculate advantage for chosen and rejected responses
    # log(P_policy(y_chosen|x) / P_ref(y_chosen|x)) = log(P_policy(y_chosen|x)) - log(P_ref(y_chosen|x))
    chosen_advantages = policy_chosen_logps - ref_chosen_logps          
    rejected_advantages = policy_rejected_logps - ref_rejected_logps
    
    # Compute the log ratio of probabilities
    logits = beta * (chosen_advantages - rejected_advantages)
    
    # Compute the DPO loss (negative log of sigmoid)
    dpo_loss = -F.logsigmoid(logits).mean()
    
    return dpo_loss


def compute_prompt_nmask(full_mask, prompt_lengths):
    prompt_nmask = torch.ones_like(full_mask.clone())[:,1:]  # (B, seq_length-1)
    for i, p_len in enumerate(prompt_lengths):
        prompt_nmask[i, :p_len-1] = 0       # p_len - 1 since the target removes the 1st token of the input_ids (which is also the 1st prompt token)
    return prompt_nmask


# Define function to compute log probabilities
def compute_logprobs(model, input_ids, attention_mask, prompt_nmask=None):

    # Pytorch Context manager which enables gradients in the enclosing block based on whether model is in training mode or not
    with torch.set_grad_enabled(model.training):
        outputs = model(input_ids=input_ids, attention_mask=attention_mask, return_dict=True)
        logits = outputs.logits[:, :-1, :]  # (B, seq_length-1, |V|) Shift to align with targets (Leave out the last <EOS> token)
        
        # Get targets (next tokens) (in teacher forcing, the first token is not needed for loss computation as it is not predicted in the targets)
        targets = input_ids[:, 1:].clone()  # (B, seq_length-1)
        
        # Create mask for valid positions (non-padding)
        target_mask = (targets != tokenizer.pad_token_id).float()
        
        # Get log probabilities
        # Computes log probabilities for each token in the vocabulary at a particular position
        log_probs = F.log_softmax(logits, dim=-1)

        # Gather log probs for the target tokens across the vocabulary dimension, with index at each position choosing the target token whose logprob is to be gathered
        token_logprobs = torch.gather(log_probs, dim=2, index=targets.unsqueeze(-1)).squeeze(-1)    # (B, seq_length-1)
        
        # Apply mask and calculate sequence log probability
        token_logprobs = token_logprobs * target_mask

        # Create copy of target mask
        final_mask = target_mask.clone()

        # Multiply with prompt nmask to mask out the prompt token logprobs
        if prompt_nmask is not None:
            token_logprobs = token_logprobs * prompt_nmask
            final_mask = final_mask * prompt_nmask
        seq_logprobs = token_logprobs.sum(dim=1) / (final_mask.sum(dim=1) + 1e-8)
        
        return seq_logprobs

In [ ]:
# For saving the model in .pt format
def save_model_checkpoint(model, path):
    torch.save(model.state_dict(), path)
    print(f"Model checkpoint saved to {path}")


# For loading the saved .pt checkpoint
def load_model_checkpoint(model, optimizer=None, scheduler=None, path=None):

    checkpoint = torch.load(path, map_location=device)
    model.load_state_dict(checkpoint)
    
    return model

In [ ]:
# DPO Training loop
best_loss = float('inf')
training_stats = []

for epoch in range(num_epochs):
    model.train()
    epoch_loss = 0
    progress_bar = tqdm(train_dataloader, desc=f"Epoch {epoch+1}/{num_epochs}")
    
    for batch in progress_bar:
        prompt_input_ids = batch["prompt_input_ids"].to(device)
        prompt_attention_mask = batch["prompt_attention_mask"].to(device)
        chosen_input_ids = batch["chosen_input_ids"].to(device)
        chosen_attention_mask = batch["chosen_attention_mask"].to(device)
        rejected_input_ids = batch["rejected_input_ids"].to(device)
        rejected_attention_mask = batch["rejected_attention_mask"].to(device)
        
        # Prepare full sequences (prompt + response)
        chosen_full_input_ids = []
        chosen_full_mask = []
        rejected_full_input_ids = []
        rejected_full_mask = []

        prompt_lengths = []
        
        # Concatenate prompts with responses for each example in batch

        for i in range(prompt_input_ids.size(0)):
            # Remove padding from prompt
            p_len = prompt_attention_mask[i].sum().item()
            prompt_lengths.append(p_len)
            prompt_input_id = prompt_input_ids[i, :p_len]
            
            # Remove padding from responses
            c_len = chosen_attention_mask[i].sum().item()
            chosen_input_id = chosen_input_ids[i, :c_len]
            
            r_len = rejected_attention_mask[i].sum().item()
            rejected_input_id = rejected_input_ids[i, :r_len]
            
            # Concatenate and pad each sequence
            chosen_full_id = torch.cat([prompt_input_id, chosen_input_id])[:max_length]
            chosen_full_mask_i = torch.ones_like(chosen_full_id)
            
            rejected_full_id = torch.cat([prompt_input_id, rejected_input_id])[:max_length]
            rejected_full_mask_i = torch.ones_like(rejected_full_id)
            
            # Append to lists
            chosen_full_input_ids.append(chosen_full_id)
            chosen_full_mask.append(chosen_full_mask_i)
            rejected_full_input_ids.append(rejected_full_id)
            rejected_full_mask.append(rejected_full_mask_i)
        
        # Pad sequences in batch
        chosen_full_input_ids = nn.utils.rnn.pad_sequence(chosen_full_input_ids, batch_first=True, padding_value=tokenizer.pad_token_id)
        chosen_full_mask = nn.utils.rnn.pad_sequence(chosen_full_mask, batch_first=True, padding_value=0)

        rejected_full_input_ids = nn.utils.rnn.pad_sequence(rejected_full_input_ids, batch_first=True, padding_value=tokenizer.pad_token_id)
        rejected_full_mask = nn.utils.rnn.pad_sequence(rejected_full_mask, batch_first=True, padding_value=0)

        chosen_full_prompt_nmask = compute_prompt_nmask(chosen_full_mask, prompt_lengths)
        rejected_full_prompt_nmask = compute_prompt_nmask(rejected_full_mask, prompt_lengths)
        
        # Compute log probabilities from policy model (being trained)
        policy_chosen_logps = compute_logprobs(model, chosen_full_input_ids, chosen_full_mask, chosen_full_prompt_nmask)
        policy_rejected_logps = compute_logprobs(model, rejected_full_input_ids, rejected_full_mask, rejected_full_prompt_nmask)
        
        # Compute log probabilities from reference model (frozen)
        with torch.no_grad():
            ref_chosen_logps = compute_logprobs(ref_model, chosen_full_input_ids, chosen_full_mask)
            ref_rejected_logps = compute_logprobs(ref_model, rejected_full_input_ids, rejected_full_mask)
        
        # Compute DPO loss
        chosen_advantages = policy_chosen_logps - ref_chosen_logps          
        rejected_advantages = policy_rejected_logps - ref_rejected_logps

        # Compute rewards
        rewards_chosen = (beta * chosen_advantages).mean()
        rewards_rejected = (beta * rejected_advantages).mean()
        rewards_accuracies = (chosen_advantages > rejected_advantages).float().mean()
        rewards_margins = rewards_chosen - rewards_rejected
        
        logits = beta * (chosen_advantages - rejected_advantages)

        loss = compute_dpo_loss(
            policy_chosen_logps, policy_rejected_logps,
            ref_chosen_logps, ref_rejected_logps,
            beta
        )
        
        # Backward and optimize
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)  # Gradient clipping
        optimizer.step()
        scheduler.step()
        
        epoch_loss += loss.item()
        progress_bar.set_postfix({"loss": loss.item()})
        
        # Log to wandb
        wandb.log({
            "batch_loss": loss.item(),
            "rewards/chosen": rewards_chosen.item(),
            "rewards/rejected": rewards_rejected.item(),
            "rewards/accuracies": rewards_accuracies,
            "rewards/margins": rewards_margins,
            "learning_rate": scheduler.get_last_lr()[0]
        })
    
    # Calculate average loss for the epoch
    avg_loss = epoch_loss / len(train_dataloader)
    
    # Save training stats
    training_stats.append({
        'epoch': epoch + 1,
        'avg_loss': avg_loss,
    })
    
    print(f"Epoch {epoch+1} average loss: {avg_loss:.4f}")
    
    # Log to wandb
    wandb.log({
        "epoch": epoch + 1,
        "avg_loss": avg_loss,
    })
    
    # Evaluate model on a sample prompt after each epoch
    model.eval()
    
    # Generate sample text
    sample_prompt = "Explain the theory of relativity:"
    input_ids = tokenizer.encode(sample_prompt, return_tensors="pt").to(device)
    
    with torch.no_grad():
        # Generate with reference model
        ref_output = ref_model.generate(
            input_ids, 
            max_length=300, 
            num_return_sequences=1,
            temperature=0.7,
            do_sample=True,
        )
        
        # Generate with tuned model
        policy_output = model.generate(
            input_ids, 
            max_length=300, 
            num_return_sequences=1,
            temperature=0.7,
            do_sample=True,
        )
    
    ref_text = tokenizer.decode(ref_output[0], skip_special_tokens=True)
    policy_text = tokenizer.decode(policy_output[0], skip_special_tokens=True)
    
    print("\nReference Model Output:")
    print(ref_text)
    print("\nTuned Model Output:")
    print(policy_text)

    # Log sample response to W&B Table
    text_table.add_data(sample_prompt, ref_text, policy_text)
    
    # Save model checkpoint if it's the best so far

    model_save_path = f"{ASSIGNMENT_PREFIX}_dpo_trained.pt"
    save_model_checkpoint(
        model, 
        optimizer, 
        scheduler, 
        epoch, 
        avg_loss, 
        model_save_path
    )

    if avg_loss < best_loss:
        best_loss = avg_loss
        # Save model in .pt format
        model_save_path = f"{ASSIGNMENT_PREFIX}_dpo_trained.pt"
        save_model_checkpoint(
            model, 
            optimizer, 
            scheduler, 
            epoch, 
            avg_loss, 
            model_save_path
        )


run.log({"training_samples" : text_table})

# Finish wandb logging
wandb.finish()

print("Training complete!")

# Task C : Evaluation

In [13]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
import numpy as np
import pandas as pd
from datasets import load_dataset
from tqdm import tqdm
import time
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from rouge_score import rouge_scorer

In [ ]:
# Load models
def load_model(base_model_path, finetuned_model_path):
    model = AutoModelForCausalLM.from_pretrained(base_model_path)
    model.load_state_dict(torch.load(finetuned_model_path))

    model.eval()
    if torch.cuda.is_available():
        model = model.to("cuda")

    return model

# Models to compare
model_name_or_path = "openai-community/gpt2-medium"
tokenizer = AutoTokenizer.from_pretrained(model_name_or_path)

rlhf_model_path = f"{ASSIGNMENT_PREFIX}_rlhf_trained.pt"
dpo_model_path = f"{ASSIGNMENT_PREFIX}_dpo_trained.pt"

print("Loading RLHF model...")
rlhf_model = load_model(model_name_or_path, rlhf_model_path)
print("Loading DPO model...")
dpo_model = load_model(model_name_or_path, dpo_model_path)

In [ ]:
# Load test dataset containing harmful/stereotype questions
print("Loading test dataset...")
test_dataset = load_dataset("csv", data_files="data/preference_test.csv", split="train")
print(len(test_dataset))

In [ ]:
# Function to generate responses
def generate_response(model, tokenizer, prompt, max_length=200):
    inputs = tokenizer(prompt, return_tensors="pt")
    if torch.cuda.is_available():
        inputs = {k: v.to("cuda") for k, v in inputs.items()}
    
    start_time = time.time()
    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_length=max_length,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
            pad_token_id=tokenizer.eos_token_id
        )
    end_time = time.time()
    
    response = tokenizer.decode(output[0], skip_special_tokens=True)
    # Remove the prompt from the response
    response = response[len(prompt):].strip()
    
    return response, end_time - start_time

In [ ]:
# Initialize metrics
scorer = rouge_scorer.RougeScorer(['rougeL'], use_stemmer=True)
smoothie = SmoothingFunction().method1

# Initialize results dictionary
results = {
    "rlhf": {
        "responses": [],
        "generation_times": [],
        "bleu_scores": [],
        "rougeL_scores": []
    },
    "dpo": {
        "responses": [],
        "generation_times": [],
        "bleu_scores": [],
        "rougeL_scores": []
    }
}

In [ ]:
# Generate responses and evaluate
for example in tqdm(test_dataset):
    prompt = example["Question"]
    reference = example["More_Prefered"]
    
    # RLHF model
    rlhf_response, rlhf_time = generate_response(rlhf_model, tokenizer, prompt)
    results["rlhf"]["responses"].append(rlhf_response)
    results["rlhf"]["generation_times"].append(rlhf_time)
    
    # DPO model
    dpo_response, dpo_time = generate_response(dpo_model, tokenizer, prompt)
    results["dpo"]["responses"].append(dpo_response)
    results["dpo"]["generation_times"].append(dpo_time)
    
    # Calculate BLEU scores
    rlhf_bleu = sentence_bleu([reference.split()], rlhf_response.split(), smoothing_function=smoothie)
    dpo_bleu = sentence_bleu([reference.split()], dpo_response.split(), smoothing_function=smoothie)
    results["rlhf"]["bleu_scores"].append(rlhf_bleu)
    results["dpo"]["bleu_scores"].append(dpo_bleu)
    
    # Calculate ROUGE scores
    rlhf_rouge = scorer.score(reference, rlhf_response)
    dpo_rouge = scorer.score(reference, dpo_response)
    
    results["rlhf"]["rougeL_scores"].append(rlhf_rouge["rougeL"].fmeasure)
    results["dpo"]["rougeL_scores"].append(dpo_rouge["rougeL"].fmeasure)

# Calculate average metrics
avg_results = {
    "rlhf": {
        "avg_generation_time": np.mean(results["rlhf"]["generation_times"]),
        "avg_bleu": np.mean(results["rlhf"]["bleu_scores"]),
        "avg_rougeL": np.mean(results["rlhf"]["rougeL_scores"])
    },
    "dpo": {
        "avg_generation_time": np.mean(results["dpo"]["generation_times"]),
        "avg_bleu": np.mean(results["dpo"]["bleu_scores"]),
        "avg_rougeL": np.mean(results["dpo"]["rougeL_scores"])
    }
}

# Print comparison results
print("\n=== Performance Comparison: RLHF vs DPO ===")
print("\n1. Response Quality (Higher is better)")
print(f"BLEU Score - RLHF: {avg_results['rlhf']['avg_bleu']:.4f}, DPO: {avg_results['dpo']['avg_bleu']:.4f}")
print(f"ROUGE-L - RLHF: {avg_results['rlhf']['avg_rougeL']:.4f}, DPO: {avg_results['dpo']['avg_rougeL']:.4f}")

print("\n2. Computation Cost (Lower is better)")
print(f"Average Generation Time - RLHF: {avg_results['rlhf']['avg_generation_time']:.4f}s, DPO: {avg_results['dpo']['avg_generation_time']:.4f}s")

# Save detailed results to CSV
results_df = pd.DataFrame({
    "Method": ["RLHF"] * len(results["rlhf"]["responses"]) + ["DPO"] * len(results["dpo"]["responses"]),
    "Response": results["rlhf"]["responses"] + results["dpo"]["responses"],
    "Generation Time (s)": results["rlhf"]["generation_times"] + results["dpo"]["generation_times"],
    "BLEU": results["rlhf"]["bleu_scores"] + results["dpo"]["bleu_scores"],
    "ROUGE-L": results["rlhf"]["rougeL_scores"] + results["dpo"]["rougeL_scores"]
})


# Function to analyze sample efficiency
def analyze_sample_efficiency(results_df, sample_percentages=[0.1, 0.25, 0.5, 0.75, 1.0]):
    """Analyze how metrics change as a function of training data size"""
    print("\n3. Sample Efficiency Analysis")
    
    for method in ["RLHF", "DPO"]:
        method_results = results_df[results_df["Method"] == method]
        print(f"\n{method} Sample Efficiency:")
        
        for metric in ["BLEU", "ROUGE-L"]:
            print(f"\n{metric} progression:")
            for pct in sample_percentages:
                sample_size = int(len(method_results) * pct)
                sampled_data = method_results.iloc[:sample_size]
                avg_score = sampled_data[metric].mean()
                print(f"  {int(pct*100)}% data: {avg_score:.4f}")

analyze_sample_efficiency(results_df)

print("\nComparison Summary:")
print("====================")
winner_quality = "RLHF" if (avg_results['rlhf']['avg_bleu'] + avg_results['rlhf']['avg_rougeL']) > (avg_results['dpo']['avg_bleu'] + avg_results['dpo']['avg_rougeL']) else "DPO"
winner_compute = "RLHF" if avg_results['rlhf']['avg_generation_time'] < avg_results['dpo']['avg_generation_time'] else "DPO"

print(f"Response Quality Winner: {winner_quality}")
print(f"Computation Efficiency Winner: {winner_compute}")


=== Performance Comparison: RLHF vs DPO ===

1. Response Quality (Higher is better)
BLEU Score - RLHF: 0.0014, DPO: 0.0002
ROUGE-1 - RLHF: 0.1370, DPO: 0.0073
ROUGE-2 - RLHF: 0.0137, DPO: 0.0000
ROUGE-L - RLHF: 0.0828, DPO: 0.0073

2. Computation Cost (Lower is better)
Average Generation Time - RLHF: 0.8439s, DPO: 1.9391s

Detailed results saved to 'rlhf_vs_dpo_results.csv'

3. Sample Efficiency Analysis

RLHF Sample Efficiency:

BLEU progression:
  10% data: 0.0013
  25% data: 0.0011
  50% data: 0.0015
  75% data: 0.0014
  100% data: 0.0014

ROUGE-1 progression:
  10% data: 0.1230
  25% data: 0.1252
  50% data: 0.1358
  75% data: 0.1367
  100% data: 0.1370

ROUGE-2 progression:
  10% data: 0.0120
  25% data: 0.0124
  50% data: 0.0138
  75% data: 0.0139
  100% data: 0.0137

ROUGE-L progression:
  10% data: 0.0748
  25% data: 0.0758
  50% data: 0.0819
  75% data: 0.0827
  100% data: 0.0828

DPO Sample Efficiency:

BLEU progression:
  10% data: 0.0001
  25% data: 0.0001
  50% data: 0.000